# Permutation Feature Importance Analysis

## Purpose
Compute permutation importance as a model-agnostic complementary metric to SHAP.
Importance is measured as the decrease in average precision (AP) when each feature
is randomly permuted across OOF validation samples.
Produces Figure 4 C/D in the manuscript.

## Method
- Model: LightGBM (binary classification), GroupKFold (5 splits, grouped by PDB ID)
- Metric: Average Precision (PR-AUC)
- Importance: permutation_importance() on validation data only (avoids optimistic bias)
- Variability: Mean +/- SD across 20 independent random seeds

## Input
- CSV table with all features and labels (output of the RMSD/labeling step)
- Feature list CSV from the global feature selection step (e.g., `features_global/ds_md.csv`)

## Output
- Per-target CSV: `feature_importance/permutation_importance_ds_md_{target}.csv`
- Figure 4 C/D: `feature_importance/permutation_top10_ds_md_combined.{png,pdf}`

## Run Order
1. Run feature selection notebook (04) to generate feature list CSVs
2. Run this notebook

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupKFold
import lightgbm as lgb

# Font settings consistent with other notebooks
plt.rcParams.update({
    "mathtext.fontset": "dejavusans",
    "mathtext.default": "regular",
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
})

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Update paths to match your local directory structure before running.
INPUT_CSV            = "/path/to/input.csv"
FEATURES_GLOBAL_DIR  = Path("/path/to/output/features_global")
OUT_DIR              = Path("/path/to/output/feature_importance")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SET  = "ds_md"           # feature set identifier
TARGET_COLS  = ["label_2p5", "label_5p0"]  # RMSD thresholds (2.5 Å and 5.0 Å)
GROUP_COL    = "pdb_id"
SEEDS        = list(range(1, 21)) # 20 seeds for reproducibility assessment
N_REPEATS    = 10                 # permutation repeats per fold (for stability)
TOP_K        = 10                 # top features to display
# ── End Configuration ──────────────────────────────────────────────────────

PALETTE = {"DockF": "#ff7f0e", "LigF": "#2ca02c", "MDF": "#1f77b4"}

LABEL_MAP = {
    "cnn_affinity": "DockF: CNN affinity",
    "cnn_pose_score": "DockF: CNN pose score",
    "vina_affinity": "DockF: Vina affinity",
    "md_contact_per_atom_mean": "MDF: contacts/atom (mean)",
    "md_contact_per_atom_slope": "MDF: contacts/atom (slope)",
    "md_contact_per_atom_std": "MDF: contacts/atom (SD)",
    "md_contact_slope": "MDF: contacts (slope)",
    "md_contact_std": "MDF: contacts (SD)",
    "md_distance_mean": "MDF: ligand-binding-site distance (mean)",
    "md_distance_min": "MDF: ligand-binding-site distance (min)",
    "md_distance_p10": "MDF: ligand-binding-site distance (p10)",
    "md_distance_std": "MDF: ligand-binding-site distance (SD)",
    "md_eelec_mean": "MDF: electrostatic energy (mean)",
    "md_eelec_slope": "MDF: electrostatic energy (slope)",
    "md_eelec_std": "MDF: electrostatic energy (SD)",
    "md_evdw_slope": "MDF: van der Waals energy (slope)",
    "md_evdw_std": "MDF: van der Waals energy (SD)",
    "md_mmgbsa_delta_total": r"MDF: MM/GBSA $\Delta G_{\mathrm{bind}}$ (mean)",
    "md_mmpbsa_delta_total": r"MDF: MM/PBSA $\Delta G_{\mathrm{bind}}$ (mean)",
    "md_mmpbsa_std": "MDF: MM/PBSA (SD)",
    "md_rg_last": "MDF: radius of gyration (final)",
    "md_rg_slope": "MDF: radius of gyration (slope)",
    "md_rg_std": "MDF: radius of gyration (SD)",
    "md_rmsd_ca_last": "MDF: CA RMSD (final)",
    "md_rmsd_ca_mean": "MDF: CA RMSD (mean)",
    "md_rmsd_ca_slope": "MDF: CA RMSD (slope)",
    "md_rmsf_protein_mean": "MDF: protein RMSF (mean)",
    "md_contact_unique_residues": "MDF: unique contact residues",
    "md_prolif_any_contact_count_mean": "MDF: any contact count (mean)",
    "md_prolif_hbond_freq": "MDF: H-bond frequency",
    "md_prolif_interaction_entropy": "MDF: interaction entropy",
}

def feature_category(name):
    if name in {"vina_affinity", "cnn_pose_score", "cnn_affinity"}:
        return "DockF"
    if str(name).startswith("md_"):
        return "MDF"
    return "LigF"

def feature_color(name):
    return PALETTE.get(feature_category(name), "#7f7f7f")

def wrap_label(label, max_length=25):
    label = label.replace("DS: ", "DockF: ").replace("MD: ", "MDF: ")
    if len(label) <= max_length:
        return label
    if "$" in label and len(label.split("$")[0].strip()) <= 15:
        return label
    for i in range(max_length, max(0, max_length - 10), -1):
        if i < len(label) and label[i] in [" ", ":"]:
            return label[:i] + "\n" + label[i + 1:].lstrip()
    return label[:max_length] + "\n" + label[max_length:]

def format_label(name):
    return wrap_label(LABEL_MAP.get(name, name))

In [ ]:
# Load data — pose rows only (consistent with training notebook)
df = pd.read_csv(INPUT_CSV)
df = df[df["ligand_id"].astype(str).str.startswith("pose")].copy()
print(f"Samples (pose rows only): {len(df)}")

# Load feature set
feat_path = FEATURES_GLOBAL_DIR / f"{FEATURE_SET}.csv"
if not feat_path.exists():
    raise FileNotFoundError(f"Feature list not found: {feat_path}\nRun notebook 04 (feature selection) first.")
features = pd.read_csv(feat_path)["feature"].astype(str).tolist()
print(f"Feature set: {FEATURE_SET} ({len(features)} features)")

missing = [f for f in features if f not in df.columns]
if missing:
    raise ValueError(f"Features missing from data: {missing[:5]}...")

In [ ]:
def train_and_permute(X, y, groups, seed):
    """
    Train LightGBM with GroupKFold and compute permutation importance
    on validation data only (avoids optimistic bias from training data).

    Returns array of shape (n_features,) — mean importance across folds.
    """
    gkf = GroupKFold(n_splits=5)
    importances = []

    for train_idx, valid_idx in gkf.split(X, y, groups):
        X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[valid_idx]

        model = lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.05,
            random_state=seed, n_jobs=-1, verbose=-1,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="average_precision",
            callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False)],
        )

        result = permutation_importance(
            model, X_va, y_va,
            scoring="average_precision",
            n_repeats=N_REPEATS,
            random_state=seed,
            n_jobs=-1,
        )
        importances.append(result.importances_mean)

    return np.mean(importances, axis=0)

In [ ]:
# Run permutation importance across all seeds and targets
results_dict = {}

for TARGET_COL in TARGET_COLS:
    print(f"\n{'='*60}")
    print(f"Target: {TARGET_COL}")
    print(f"{'='*60}")

    X = df[features]
    y = df[TARGET_COL]
    groups = df[GROUP_COL]

    seed_importances = []
    for seed in SEEDS:
        print(f"  Seed {seed:2d}...", end=" ", flush=True)
        imp = train_and_permute(X, y, groups, seed)
        seed_importances.append(imp)
        print("done")

    all_imp = np.vstack(seed_importances)
    df_imp = pd.DataFrame({
        "feature": features,
        "importance_mean": all_imp.mean(axis=0),
        "importance_sd":   all_imp.std(axis=0),
    }).sort_values("importance_mean", ascending=False)

    csv_path = OUT_DIR / f"permutation_importance_{FEATURE_SET}_{TARGET_COL}.csv"
    df_imp.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path.name}")

    results_dict[TARGET_COL] = df_imp.copy()

print("\nAll targets processed.")

In [ ]:
# ── Combined 2-panel plot (Figure 4 C/D) ───────────────────────────────────
# Compute common x-axis limits across both RMSD thresholds
xmaxs = []
for tgt in TARGET_COLS:
    top = results_dict[tgt].head(TOP_K)
    xmaxs.append((top["importance_mean"] + top["importance_sd"].fillna(0)).max())
x_max = float(np.max(xmaxs))
common_xlim = (0.0, x_max + max(0.05 * x_max, 0.01))
print(f"[INFO] Common x-axis limit: {common_xlim}")

fig, axes = plt.subplots(1, 2, figsize=(18, 8.5))
handles = [plt.Rectangle((0, 0), 1, 1, color=PALETTE[k]) for k in ["DockF", "MDF"]]

for idx, tgt in enumerate(TARGET_COLS):
    ax = axes[idx]
    top = results_dict[tgt].head(TOP_K).iloc[::-1]

    ax.barh(range(len(top)), top["importance_mean"],
            xerr=top["importance_sd"].fillna(0.0), capsize=4,
            color=[feature_color(f) for f in top["feature"]],
            edgecolor="black", linewidth=0.5)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([format_label(f) for f in top["feature"]], fontsize=16)
    ax.set_xlabel("Permutation importance ± SD\n(decrease in average precision)", fontsize=16)
    ax.tick_params(axis="x", labelsize=14)
    rmsd_label = "2.5 Å" if tgt == "label_2p5" else "5.0 Å"
    ax.set_title(f"RMSD {rmsd_label}", fontsize=17, pad=12)
    ax.grid(axis="x", alpha=0.3)
    ax.set_xlim(common_xlim)

fig.legend(handles, ["DockF", "MDF"],
           loc="lower center", bbox_to_anchor=(0.5, 0.03),
           ncol=2, frameon=True, fontsize=17)
plt.tight_layout(pad=2.5, rect=[0, 0.07, 1, 1])

base = OUT_DIR / f"permutation_top{TOP_K}_{FEATURE_SET}_combined"
fig.savefig(str(base) + ".png", dpi=300, bbox_inches="tight")
fig.savefig(str(base) + ".pdf", bbox_inches="tight")
plt.show()
print(f"[OK] Saved Figure 4 C/D: {base.name}.*")